# **Desarrollo del taller 3 — Mini-Proyecto de clasificación de texto con BERT**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yeavil-neny/NLP-ProcesamientoLeguajeNatural/blob/main/Taller_3/proyecto_bert_clasificacion.ipynb)

**Grupo:** Delta

## **Definición del problema**

**🎯 Objetivo del notebook:** resolver **el mismo problema de los Talleres 1 y 2** — clasificar la satisfacción de clientes de e-commerce a partir de sus reseñas en español, en una escala de **1 a 5 estrellas** — pero ahora con un **modelo pre-entrenado de tipo BERT** y las herramientas de **Hugging Face**, siguiendo la estructura del notebook guía `1-text-classification-with-hf.ipynb`.

### **El problema, bien planteado**

Las plataformas de e-commerce reciben miles de reseñas al día. Analizarlas a mano es imposible, y el "promedio de estrellas" esconde información valiosa: no es lo mismo una reseña de 2 estrellas ("no funciona bien, pero llegó rápido") que una de 5. Queremos un sistema que **lea el texto de la reseña y prediga cuántas estrellas le daría el cliente**, para poder:

1. **Detectar insatisfacción temprana:** alertar al vendedor cuando hay reseñas negativas que merecen respuesta rápida.
2. **Triaje automático:** enrutar reseñas críticas (1-2★) a servicio al cliente y reseñas positivas (4-5★) a marketing.
3. **Auditar valoraciones:** comparar la estrella que el usuario marcó con la que "dice" su texto; discrepancias grandes pueden indicar valoraciones erróneas.

Este es un problema **difícil de NLP por naturaleza**: las 5 clases son **ordinales** (1★ está más cerca de 2★ que de 5★), los textos son **cortos, con errores ortográficos, lenguaje informal y sin contexto del producto**, y las clases intermedias (2-4★) son semánticamente ambiguas: mezclan quejas y elogios en un mismo texto.

### **¿Por qué BERT para este problema? (y por qué ahora)**

En el **Taller 1** una LSTM alcanzó ~0.52 de accuracy y en el **Taller 2** nuestro Transformer implementado desde cero obtuvo ~0.49 — peor que la LSTM, como documentamos, porque con 50.000 textos cortos un transformer "en blanco" no logra aprender representaciones ricas. Nuestra hipótesis al cierre del Taller 2 fue precisamente que **los transformers brillan cuando parten de un pre-entrenamiento**. Este taller es el experimento que la pone a prueba.

1. **Transfer learning:** BETO (BERT en español, de la U. de Chile) ya "leyó" un corpus masivo de español. Le ahorramos al modelo aprender el idioma desde cero: solo debe especializarse en reseñas.
2. **Contexto bidireccional:** a diferencia de la LSTM, BERT lee cada token mirando TODO el texto a la vez (izquierda y derecha), clave en reseñas con contrastes ("llegó tarde, **pero** la calidad es excelente").
3. **Tres estrategias comparables:** el notebook guía muestra featurizer con cabeza lineal, featurizer con clasificador propio y fine-tuning. Las replicaremos con nuestros datos y añadiremos una cuarta: **embeddings + clasificador clásico (regresión logística)**, que la guía menciona pero no ejecuta.

### **Hipótesis del experimento**

- **H1:** el fine-tuning de BETO superará claramente a los modelos de los Talleres 1 y 2 (> 0.52 de accuracy).
- **H2:** incluso con el encoder congelado (solo entrenando la cabeza), BETO superará a los modelos entrenados desde cero — evidencia del valor del pre-entrenamiento.
- **H3:** los errores se concentrarán entre **estrellas vecinas** (predecir 3★ cuando la real es 2★), como vimos en los talleres anteriores; mediremos la "distancia en estrellas" de los errores.

### **Mapa del notebook**

1. Setup y carga de datos (mismo dataset y muestreo que T1 y T2 → comparación justa).
2. **EDA** de las reseñas (balance, longitudes, nubes de palabras 1★ vs 5★).
3. **Tokenizador BETO** y preparación de los datos.
4. **Experimento 1:** BETO congelado + cabeza lineal (featurizer).
5. **Experimento 2:** BETO congelado + clasificador MLP propio.
6. **Experimento 3:** fine-tuning completo.
7. **Experimento 4 (más allá de la guía):** embeddings de BETO + regresión logística.
8. Análisis del mejor modelo: matriz de confusión, distancia de errores, análisis cualitativo.
9. Demo de predicción y comparativa final T1 vs T2 vs T3.

> **⏱️ Tiempos esperados en GPU T4 de Colab:** Exp. 1 y 2 ≈ 10-15 min c/u, Exp. 3 ≈ 20-25 min, Exp. 4 ≈ 10 min, resto ≈ 5 min. **Total ≈ 1 hora.** Activa la GPU: `Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU`.

#### Referencias
- Dataset: [SetFit/amazon_reviews_multi_es](https://huggingface.co/datasets/SetFit/amazon_reviews_multi_es)
- Modelo: [BETO — dccuchile/bert-base-spanish-wwm-cased](https://huggingface.co/dccuchile/bert-base-spanish-wwm-cased)
- [BERT: Pre-training of Deep Bidirectional Transformers](https://arxiv.org/abs/1810.04805)
- [Documentación de Hugging Face Transformers](https://huggingface.co/docs/transformers)
- Notebooks guía del curso: `1-text-classification-with-hf.ipynb` (y su versión comentada en este mismo directorio)

In [ ]:
# Setup: detectamos el entorno y silenciamos avisos
import warnings

warnings.filterwarnings('ignore')

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print('¿Ejecutando en Google Colab?', IN_COLAB)

In [ ]:
if IN_COLAB:
    # -q = quiet (instala sin imprimir demasiado)
    !pip install -q transformers datasets evaluate torchinfo accelerate wordcloud

In [ ]:
# Importaciones principales
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn

from datasets import load_dataset, DatasetDict
from transformers import (AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
                           Trainer, TrainingArguments, set_seed)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from tqdm.auto import tqdm

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Reproducibilidad: fijamos la semilla en Python/numpy/PyTorch.
# Usamos la MISMA semilla (42) y el MISMO muestreo que en los Talleres 1 y 2
# para que la comparación entre modelos sea justa.
set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')
if device.type == 'cuda':
    print(f'GPU disponible: {torch.cuda.get_device_name(0)}')

## **Carga de Datos**

**🎯 Objetivo de la sección:** traer el dataset de reseñas de Amazon en español con **exactamente el mismo muestreo de los Talleres 1 y 2** (50.000 entrenamiento / 5.000 validación / 5.000 test, barajado con semilla 42). Cambiar los datos entre talleres invalidaría cualquier comparación entre modelos.

In [ ]:
# Cargamos el dataset de reseñas de Amazon en español (mismo del Taller 1 y 2)
print("Descargando el dataset 'SetFit/amazon_reviews_multi_es'...")
dataset = load_dataset("SetFit/amazon_reviews_multi_es")

# Mismo submuestreo de los talleres anteriores (semilla fija -> reproducible)
subset_size_train = 50000
subset_size_eval = 5000

train_data = dataset['train'].shuffle(seed=42).select(range(subset_size_train))
val_data = dataset['validation'].shuffle(seed=42).select(range(subset_size_eval))
test_data = dataset['test'].shuffle(seed=42).select(range(subset_size_eval))

# Nombres bonitos para las clases (0..4 -> 1..5 estrellas)
id2label = {0: '1★', 1: '2★', 2: '3★', 3: '4★', 4: '5★'}
label2id = {v: k for k, v in id2label.items()}
NUM_CLASSES = 5

print(f"\n--- Resumen de Datos ---")
print(f"Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")
print("\n--- Ejemplo de un registro ---")
print(f"Texto: {train_data[0]['text'][:250]}...")
print(f"Etiqueta: {train_data[0]['label']} (escala 0-4, donde 0 es 1 estrella)")

## **Análisis Exploratorio de Datos (EDA)**

**🎯 Objetivo de la sección:** conocer la forma de nuestros datos antes de entrenar:
1. ¿Están balanceadas las 5 clases?
2. ¿Qué tan largos son los textos (en palabras y en tokens)?
3. ¿Qué palabras distinguen a las reseñas muy negativas de las muy positivas?

Estas respuestas definen decisiones del modelo (longitud máxima de secuencia, métrica a usar).

In [ ]:
# Convertimos el train set a DataFrame para explorarlo con pandas
df_train = train_data.to_pandas()
df_train['estrellas'] = df_train['label'] + 1   # pasamos de 0-4 a 1-5 para leer mejor
df_train.head()

In [ ]:
# --- 1. Balance de clases ---
conteo = df_train['estrellas'].value_counts().sort_index()
print(conteo)

ax = conteo.plot.bar(rot=0, color=sns.color_palette('viridis', 5))
ax.set_title('Distribución de estrellas en el set de entrenamiento')
ax.set_xlabel('Estrellas')
ax.set_ylabel('Número de reseñas')
plt.show()

In [ ]:
# --- 2. Ejemplos textuales por categoría (para calibrar la intuición) ---
for estrella in range(1, 6):
    print("=" * 70)
    print(f"RESEÑAS DE {estrella} ESTRELLA(S):")
    ejemplos = df_train[df_train['estrellas'] == estrella]['text'].head(3)
    for e in ejemplos:
        print(f"  - \"{e[:150]}\"")

In [ ]:
# --- 3. Longitud de los textos en PALABRAS por clase ---
df_train['Palabras por Texto'] = df_train['text'].str.split().apply(len)

df_train.boxplot('Palabras por Texto', by='estrellas', grid=False,
                 showfliers=False, color='black', figsize=(10, 5))
plt.suptitle('')
plt.title('Longitud (en palabras) por cantidad de estrellas')
plt.xlabel('Estrellas')
plt.show()

print('Mediana de palabras por clase:')
print(df_train.groupby('estrellas')['Palabras por Texto'].median())

In [ ]:
# --- 4. Nubes de palabras: ¿qué dicen los clientes muy insatisfechos vs muy satisfechos? ---
from wordcloud import WordCloud
from nltk.corpus import stopwords
import nltk

nltk.download('stopwords', quiet=True)
sw = set(stopwords.words('spanish'))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, (estrella, titulo) in zip(axes, [(1, '1★ (muy insatisfecho)'), (5, '5★ (muy satisfecho)')]):
    textos = ' '.join(df_train[df_train['estrellas'] == estrella]['text'].str.lower())
    wc = WordCloud(width=700, height=350, background_color='white',
                   stopwords=sw, collocations=False).generate(textos)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(titulo)
    ax.axis('off')
plt.show()

# Observa el contraste: en 1★ dominan "no", "mal", "dinero", "devolución"...
# mientras en 5★ abundan "bien", "calidad", "recomiendo", "buena".

**Observaciones del EDA:**

- **Balance de clases:** las 5 categorías tienen ~10.000 ejemplos cada una → dataset balanceado, el accuracy será una métrica confiable (igual que concluímos en los Talleres 1 y 2 con este mismo dataset).
- **Longitudes:** las reseñas son **cortas** (mediana ≈ 10-20 palabras) frente a las noticias del notebook guía (~500). Esto tiene dos consecuencias: (1) podemos usar secuencias mucho más cortas que 512 y ahorrar muchísimo tiempo de cómputo, y (2) hay poco contexto por reseña, lo que explica por qué a los modelos entrenados desde cero les costó tanto en los talleres anteriores.
- **Vocabulario por clase:** las nubes muestran lenguaje claramente polarizado en los extremos (1★ vs 5★), pero las clases intermedias comparten palabras de ambos mundos — anticipo de la confusión entre estrellas vecinas que analizaremos en los errores.

## **El tokenizador BETO**

**🎯 Objetivo de la sección:** cargar el tokenizador del checkpoint pre-entrenado y decidir la longitud máxima de secuencia (`MAX_LEN`) basándonos en datos.

Regla de oro del notebook guía: **el tokenizador y el modelo deben ser del mismo paquete**. El modelo que usaremos es **BETO** (`dccuchile/bert-base-spanish-wwm-cased`), el BERT entrenado en español por la Universidad de Chile: la misma arquitectura que BERT, pero con vocabulario y pesos "hispanohablantes", lo cual es justo lo que nuestro problema necesita.

In [ ]:
MODEL_CKPT = "dccuchile/bert-base-spanish-wwm-cased"

# AutoTokenizer detecta solo qué tipo de tokenizador corresponde al checkpoint
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

# El token de relleno [PAD] sirve para que todos los textos midan lo mismo
if tokenizer.pad_token is None:
    tokenizer.pad_token = '[PAD]'

# Prueba con una reseña típica del dominio:
ejemplo = "¡El producto llegó tarde y no funciona, una pérdida de dinero!"
enc = tokenizer(ejemplo, truncation=True, max_length=16, padding='max_length')
print('Tokens   :', tokenizer.convert_ids_to_tokens(enc['input_ids']))
print('input_ids:', enc['input_ids'])
print('attention_mask:', enc['attention_mask'])
print('vocab_size:', tokenizer.vocab_size, '| model_max_length:', tokenizer.model_max_length)

Fíjate en dos cosas: el tokenizador maneja bien el español (acentos, ¡!) usando *subpalabras* cuando hace falta (los trozos que empiezan con `##` van pegados al anterior), y el `attention_mask` marca con `0` dónde empieza el relleno `[PAD]` para que el modelo lo ignore.

Ahora medimos la longitud **en tokens BETO** sobre una muestra, para elegir `MAX_LEN` con evidencia:

In [ ]:
# Distribución de longitudes en TOKENS sobre una muestra del train set
muestra = train_data.shuffle(seed=42).select(range(2000))
lens = [len(tokenizer(t)['input_ids']) for t in muestra['text']]
lens = np.array(lens)

print('Percentiles de longitud en tokens:')
for p in [50, 90, 95, 99, 100]:
    print(f'  p{p}: {np.percentile(lens, p):.0f} tokens')

# Elegimos 128: cubre >99% del corpus con holgura y es una fracción del
# máximo de BERT (512) -> menos memoria y entrenamientos más rápidos.
MAX_LEN = 128
cobertura = (lens <= MAX_LEN).mean()
print(f'\nCon MAX_LEN={MAX_LEN} se cubre el {cobertura*100:.2f}% de las reseñas (el resto se trunca)')

## **Preparación de los datos**

**🎯 Objetivo de la sección:** tokenizar los tres conjuntos una sola vez. A diferencia del notebook guía (que usaba 512 tokens para noticias largas), aquí tokenizamos a `MAX_LEN=128` — con reseñas de ~20 tokens de mediana, 128 sobra y acelera todo.

La tokenización se hace con `map(..., batched=True)`: la función recibe un lote de textos y devuelve sus `input_ids` y `attention_mask`, que se añaden como columnas nuevas al dataset. La columna `label` (0-4) ya viene numérica en este dataset, así que no hay que mapear nada más.

In [ ]:
# DatasetDict con los tres conjuntos (mismo objeto que usa la guía)
splits = DatasetDict({'train': train_data, 'val': val_data, 'test': test_data})

def tokenize_fn(examples):
    # truncation=True -> corta textos más largos que MAX_LEN
    # padding='max_length' -> rellena los más cortos con [PAD]
    return tokenizer(examples['text'], truncation=True, max_length=MAX_LEN, padding='max_length')

tokenized = splits.map(tokenize_fn, batched=True)
tokenized

## **Métricas de calidad**

**🎯 Objetivo de la sección:** definir cómo vamos a juzgar a los modelos ANTES de entrenarlos.

- **Accuracy** (exactitud): % de reseñas con la estrella exacta correcta. Es la métrica usada en los Talleres 1 y 2 → la base de la comparación.
- **F1 macro**: promedia el F1 de las 5 clases por igual. Si un modelo ignora la clase 2★ (típico), el accuracy puede disimularlo, pero el F1 macro no.

La función `compute_metrics` será llamada por el `Trainer` al final de cada época con las predicciones de validación.

In [ ]:
def compute_metrics(pred):
    """Calcula las métricas a partir de los logits del modelo.

    pred.label_ids     -> etiquetas reales
    pred.predictions   -> matriz [n_ejemplos, 5] con los logits por clase
    """
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)   # clase con el puntaje más alto
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro'),
    }

# Configuración global de entrenamiento (igual para los 4 experimentos)
BATCH_SIZE = 32 if torch.cuda.is_available() else 8

def make_training_args(output_dir, epochs, lr):
    """Hiperparámetros comunes. Cada experimento elige épocas y learning rate."""
    return TrainingArguments(
        output_dir=output_dir,          # carpeta de checkpoints y logs
        num_train_epochs=epochs,
        learning_rate=lr,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=64,
        weight_decay=0.01,              # regularización contra sobreajuste
        eval_strategy='epoch',         # evaluamos en validación cada época
        save_strategy='epoch',         # y guardamos checkpoint cada época
        save_total_limit=1,             # solo el mejor checkpoint (ahorro de disco)
        load_best_model_at_end=True,    # al final quedamos con el mejor modelo
        metric_for_best_model='accuracy',
        fp16=IN_COLAB,                  # precisión mixta: más rápido en GPU
        logging_steps=100,              # frecuencia de logs del entrenamiento
        report_to='tensorboard',
        seed=42,
    )

# Aquí iremos acumulando los resultados de los 4 experimentos
resultados = {}

## **Experimento 1: BETO congelado como featurizer + cabeza lineal**

**🎯 Objetivo de la sección:** la estrategia más barata. Congelamos el encoder BETO completo (`requires_grad=False`: sus 110 millones de pesos NO se re-entrenan) y entrenamos únicamente la cabeza lineal de 768→5 que Hugging Face añade al cargar el modelo con `AutoModelForSequenceClassification`.

> **💡 Qué esperar:** el modelo "ve" las reseñas con el conocimiento de español que ya trae BETO, pero sus capas internas nunca se adaptan a nuestro dominio. Aun así, el embedding de [CLS] ya condensa mucho significado; una simple capa lineal encima puede ir sorprendentemente lejos. La pregunta es: ¿le alcanza para superar a la LSTM del Taller 1 (0.5168)?

> **💡 Learning rate:** usamos `1e-3`, mayor que el `2e-5` de la guía. Motivo: aquí solo se entrena la cabeza (pesos nuevos, aleatorios), que aprende rápido; `2e-5` es para cuando se ajustan pesos PRE-entrenados que conviene mover apenas.

In [ ]:
set_seed(42)

# Modelo nuevo + cabeza de clasificación con 5 salidas (una por estrella)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT, num_labels=NUM_CLASSES, id2label=id2label, label2id=label2id
).to(device)

# CONGELAMOS el encoder: el modelo base no se re-entrena (featurizer)
for param in model.base_model.parameters():
    param.requires_grad = False

# Resumen: notar la columna 'trainable' -> solo el clasificador se entrena
from torchinfo import summary
dummy = tokenizer('hola', max_length=MAX_LEN, truncation=True, padding='max_length', return_tensors='pt')
with torch.no_grad():
    print(summary(model, input_size=[dummy['input_ids'].shape] * 3,
                  dtypes=[dummy['input_ids'].dtype] * 3, depth=2,
                  col_names=['num_params', 'trainable']))

In [ ]:
%%time
training_args = make_training_args('./hf/exp1-featurizer-lineal', epochs=3, lr=1e-3)
trainer_exp1 = Trainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['val'],
    processing_class=tokenizer,
)
trainer_exp1.train()

In [ ]:
# Evaluación en el conjunto de TEST (examen final: datos nunca vistos)
model.eval()
res_exp1 = trainer_exp1.evaluate(tokenized['test'])
resultados['E1: featurizer + lineal'] = res_exp1
res_exp1

## **Experimento 2: BETO congelado + clasificador MLP propio**

**🎯 Objetivo de la sección:** misma estrategia del experimento 1 (encoder congelado), pero reemplazamos la cabeza lineal simple por un **perceptrón multicapa (MLP)** más profundo: 768 → 512 → 256 → 5, con ReLU y Dropout.

> **💡 Detalle técnico:** la guía añade `LogSoftmax` al final de su clasificador, pero el `Trainer` de HF calcula la pérdida con `CrossEntropyLoss`, que **ya incluye su propio softmax**. Poner `LogSoftmax` antes produce un "doble log-softmax" que degrada la pérdida sin cambiar la clase elegida. En nuestro proyecto **lo omitimos deliberadamente** — y queda como nota de discusión para la clase.

> **💡 Qué esperar:** en la guía la mejora fue marginal (80.3% vs 80.6% en noticias). Aquí lo ponemos a prueba con nuestras reseñas. La intuición: con el encoder congelado, la capacidad extra de la cabeza debería ayudar poco, porque el cuello de botella no es la cabeza sino un encoder no adaptado al dominio.

In [ ]:
set_seed(42)
# Liberamos memoria del modelo anterior antes de cargar uno nuevo
del model
torch.cuda.empty_cache()

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT, num_labels=NUM_CLASSES, id2label=id2label, label2id=label2id
).to(device)
for param in model.base_model.parameters():
    param.requires_grad = False

# Nuestro clasificador: tubería de capas que se ejecutan en orden.
# OJO: sin LogSoftmax final (ver nota de arriba).
model.classifier = nn.Sequential(
    nn.Linear(768, 512),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(256, NUM_CLASSES),
)
print(model.classifier)

In [ ]:
%%time
training_args = make_training_args('./hf/exp2-featurizer-mlp', epochs=3, lr=1e-3)
trainer_exp2 = Trainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['val'],
    processing_class=tokenizer,
)
trainer_exp2.train()

In [ ]:
model.eval()
res_exp2 = trainer_exp2.evaluate(tokenized['test'])
resultados['E2: featurizer + MLP propio'] = res_exp2
res_exp2

## **Experimento 3: Fine-tuning completo**

**🎯 Objetivo de la sección:** la estrategia más potente del notebook guía: cargar el modelo **sin congelar nada** y re-entrenar TODAS sus capas (los ~110 millones de parámetros) sobre nuestras reseñas.

**Ajustes prudentes del fine-tuning** (los mismos de la guía):
- **Learning rate pequeño (`2e-5`):** los pesos pre-entrenados ya codifican español; pasos grandes los destruirían en vez de especializarlos.
- **Solo 2 épocas:** con esta cantidad de datos el modelo ajusta rápido; más épocas arriesgan memorizar el train set (sobreajuste).
- `load_best_model_at_end=True` nos rescata la mejor época según validación.

> **💡 Qué esperar:** aquí es donde el guion del Taller 2 ("los transformers brillan con pre-entrenamiento") debería cumplirse: BETO ajustará sus capas internas al vocabulario de las reseñas ("no funciona", "llegó roto", "recomendado") y esperamos un salto claro frente a los experimentos 1 y 2 y frente a la LSTM del Taller 1.

In [ ]:
set_seed(42)
del model
torch.cuda.empty_cache()

# Modelo NUEVO desde el checkpoint. Esta vez NO congelamos nada:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT, num_labels=NUM_CLASSES, id2label=id2label, label2id=label2id
).to(device)

training_args = make_training_args('./hf/exp3-finetuning', epochs=2, lr=2e-5)
trainer_exp3 = Trainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['val'],
    processing_class=tokenizer,
)

In [ ]:
%%time
trainer_exp3.train()

In [ ]:
model.eval()
res_exp3 = trainer_exp3.evaluate(tokenized['test'])
resultados['E3: fine-tuning completo'] = res_exp3
res_exp3

## **Experimento 4 (más allá de la guía): embeddings de BETO + regresión logística**

**🎯 Objetivo de la sección:** la guía menciona que con un modelo congelado *"no es necesario que utilicemos deep learning para la clasificación final, podemos usar algoritmos clásicos"* — pero no lo ejecuta. Nosotros sí: extraemos el embedding del token `[CLS]` (el "resumen" de la reseña, 768 números) para cada ejemplo, y entrenamos una **regresión logística** de sklearn encima. Con esto comparamos el pipeline deep learning vs. un clasificador clásico sobre las mismas características.

Además, visualizamos los embeddings con **PCA** (proyección a 2D): si las estrellas forman "grupos" separados en el espacio de embeddings, la geometría del modelo lo capturó; si se mezclan, entendemos por qué el clasificador se confunde entre estrellas vecinas.

In [ ]:
set_seed(42)
del model
torch.cuda.empty_cache()

# AutoModel devuelve SOLO el encoder (sin cabeza de clasificación)
bert = AutoModel.from_pretrained(MODEL_CKPT).to(device).eval()

def extraer_cls(ds, batch_size=64):
    """Extrae el embedding [CLS] (768 números) de cada reseña del dataset."""
    embeddings, labels = [], []
    for i in tqdm(range(0, len(ds), batch_size), desc='Extrayendo embeddings'):
        chunk = ds[i:i + batch_size]
        batch = {
            'input_ids': torch.tensor(chunk['input_ids'], device=device),
            'attention_mask': torch.tensor(chunk['attention_mask'], device=device),
        }
        with torch.no_grad():                 # no entrenamos: no hace falta gradiente
            out = bert(**batch)
        # last_hidden_state: [batch, MAX_LEN, 768] -> nos quedamos con la posición 0 ([CLS])
        embeddings.append(out.last_hidden_state[:, 0, :].float().cpu().numpy())
        labels.append(np.array(chunk['label']))
    return np.concatenate(embeddings), np.concatenate(labels)

X_train_cls, y_train_cls = extraer_cls(tokenized['train'])
X_test_cls, y_test_cls = extraer_cls(tokenized['test'])
print('Embeddings de train:', X_train_cls.shape, '| de test:', X_test_cls.shape)

In [ ]:
# Visualización PCA: ¿cómo se ven las 5 estrellas en el espacio de embeddings?
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
# Proyectamos una muestra del test para que el gráfico sea legible
idx = np.random.RandomState(42).choice(len(X_test_cls), 2000, replace=False)
P = pca.fit_transform(X_test_cls[idx])

plt.figure(figsize=(8, 6))
for estrella in range(5):
    m = y_test_cls[idx] == estrella
    plt.scatter(P[m, 0], P[m, 1], s=8, alpha=0.5, label=id2label[estrella])
plt.legend(title='Estrellas reales')
plt.title('Embeddings [CLS] de BETO proyectados con PCA (test set)')
plt.xlabel('Componente 1'); plt.ylabel('Componente 2')
plt.show()

# Si en el gráfico ves 1★ y 5★ en zonas opuestas pero 2-4★ mezcladas en el
# medio, acabas de VER por qué la confusión entre estrellas vecinas es difícil.

In [ ]:
%%time
# Clasificador clásico sobre los embeddings (congelados) de BETO
logreg = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
logreg.fit(X_train_cls, y_train_cls)

pred_lr = logreg.predict(X_test_cls)
res_exp4 = {
    'eval_accuracy': accuracy_score(y_test_cls, pred_lr),
    'eval_f1_macro': f1_score(y_test_cls, pred_lr, average='macro'),
}
resultados['E4: embeddings + regresión logística'] = res_exp4
res_exp4

## **Comparación de los 4 experimentos (y de los 3 talleres)**

**🎯 Objetivo de la sección:** poner todos los números sobre la mesa. Primero los 4 experimentos de este taller; luego la comparación con los modelos entrenados **desde cero** en los Talleres 1 y 2, sobre exactamente los mismos datos (50k/5k/5k, semilla 42) — esta es la razón de ser del experimento.

In [ ]:
# Tabla y gráfico de los 4 experimentos de este taller
df_resultados = pd.DataFrame({
    'accuracy': {k: v['eval_accuracy'] for k, v in resultados.items()},
    'f1_macro': {k: v['eval_f1_macro'] for k, v in resultados.items()},
}).T

display(df_resultados.style.format('{:.4f}').background_gradient(cmap='Greens', axis=0))

ax = df_resultados['accuracy'].sort_values().plot.barh(color='steelblue')
ax.axvline(0.5168, color='red', linestyle='--', label='Mejor modelo T1 (LSTM): 0.5168')
ax.axvline(0.2, color='gray', linestyle=':', label='Azar (5 clases): 0.20')
ax.set_title('Accuracy en test de los experimentos del Taller 3')
ax.legend(loc='lower right')
plt.show()

In [ ]:
# Comparativa histórica: Taller 1 (LSTM), Taller 2 (Transformer desde cero) y Taller 3 (BETO)
# Los valores de T1 y T2 son los medidos en las entregas anteriores (mismos datos).
comparativa = pd.DataFrame([
    {'Taller': 'T1', 'Modelo': 'LSTM simple (desde cero)',               'Accuracy test': 0.5168, 'F1 macro': None},
    {'Taller': 'T1', 'Modelo': 'LSTM óptima (desde cero)',               'Accuracy test': 0.5120, 'F1 macro': None},
    {'Taller': 'T1', 'Modelo': 'LSTM bidireccional (desde cero)',        'Accuracy test': 0.5150, 'F1 macro': None},
    {'Taller': 'T2', 'Modelo': 'Transformer desde cero',                 'Accuracy test': 0.4896, 'F1 macro': 0.4920},
    {'Taller': 'T3', 'Modelo': 'E1: featurizer + lineal',                'Accuracy test': resultados['E1: featurizer + lineal']['eval_accuracy'],   'F1 macro': resultados['E1: featurizer + lineal']['eval_f1_macro']},
    {'Taller': 'T3', 'Modelo': 'E2: featurizer + MLP propio',             'Accuracy test': resultados['E2: featurizer + MLP propio']['eval_accuracy'], 'F1 macro': resultados['E2: featurizer + MLP propio']['eval_f1_macro']},
    {'Taller': 'T3', 'Modelo': 'E3: fine-tuning completo',               'Accuracy test': resultados['E3: fine-tuning completo']['eval_accuracy'],    'F1 macro': resultados['E3: fine-tuning completo']['eval_f1_macro']},
    {'Taller': 'T3', 'Modelo': 'E4: embeddings + regresión logística',   'Accuracy test': resultados['E4: embeddings + regresión logística']['eval_accuracy'], 'F1 macro': resultados['E4: embeddings + regresión logística']['eval_f1_macro']},
])

display(comparativa.style.format({'Accuracy test': '{:.4f}', 'F1 macro': '{:.4f}'}, na_rep='—'))

## **Análisis del mejor modelo**

**🎯 Objetivo de la sección:** elegir el mejor experimento y diseccionarlo: matriz de confusión, F1 por clase, **distancia de los errores en estrellas** y lectura cualitativa de errores.

> Si al ejecutar la comparación el mejor experimento **no** fuera el E3 (fine-tuning), ajusta la celda siguiente para analizar el que haya ganado (reemplaza `trainer_exp3` y `model` por los del experimento ganador).

In [ ]:
# Usamos el modelo del Experimento 3 (fine-tuning), esperado ganador
preds = trainer_exp3.predict(tokenized['test'])
pred_ids = np.argmax(preds.predictions, axis=-1)
y_true = preds.label_ids

# F1 por clase y resumen
print(classification_report(y_true, pred_ids, target_names=[id2label[i] for i in range(5)], digits=3))

# Matriz de confusión
cm = confusion_matrix(y_true, pred_ids)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=list(id2label.values()), yticklabels=list(id2label.values()))
plt.xlabel('Predicción'); plt.ylabel('Real')
plt.title('Matriz de confusión — BETO fine-tuned')
plt.show()

In [ ]:
# Distancia de los errores: ¿el modelo falla 'por poco' o 'sin sentido'?
distancias = np.abs(y_true - pred_ids)

tabla_dist = pd.DataFrame({
    'Situación': ['Acierto exacto', 'Error de ±1 estrella', 'Error de ±2 estrellas', 'Error de ±3 o más'],
    'Reseñas': [
        (distancias == 0).sum(),
        (distancias == 1).sum(),
        (distancias == 2).sum(),
        (distancias >= 3).sum(),
    ],
})
tabla_dist['% del test'] = (tabla_dist['Reseñas'] / len(y_true) * 100).round(2)
display(tabla_dist)

acc_exacta = (distancias == 0).mean()
acc_tolerante = (distancias <= 1).mean()   # 'accuracy tolerante': acierto o error de 1 estrella
print(f'Accuracy exacta:            {acc_exacta:.4f}')
print(f'Accuracy tolerante (±1★):   {acc_tolerante:.4f}  <- la métrica que importaría en producción')

In [ ]:
# Análisis cualitativo: leamos los errores del modelo
df_test = pd.DataFrame({
    'texto': test_data['text'],
    'real': y_true,
    'predicha': pred_ids,
})
df_test['correcto'] = df_test['real'] == df_test['predicha']

errores = df_test[~df_test['correcto']]
print(f'Errores: {len(errores)} de {len(df_test)}')
print()
# Errores 'groseros' (3+ estrellas de distancia): los más interesantes
errores_graves = errores[(errores['real'] - errores['predicha']).abs() >= 3]
print(f'--- Errores graves (≥3 estrellas de distancia): {len(errores_graves)} ---')
for _, row in errores_graves.head(8).iterrows():
    print(f"  Real: {id2label[row['real']]} | Predicha: {id2label[row['predicha']]}")
    print(f"  \"{row['texto'][:180]}\"")
    print()

## **Demo: prediciendo estrellas de reseñas nuevas**

**🎯 Objetivo de la sección:** usar el modelo como se usaría "en producción": le pasamos textos arbitrarios y devuelve la predicción con sus probabilidades. Prueba con tus propias reseñas — reescribe la lista y vuelve a ejecutar.

In [ ]:
def predecir_estrellas(texto, model, tokenizer):
    """Predice las estrellas (0-4) de una reseña y devuelve probabilidades."""
    inputs = tokenizer(texto, truncation=True, max_length=MAX_LEN,
                      padding='max_length', return_tensors='pt').to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
    return int(logits.argmax()), probs

# El mejor modelo de los experimentos 1-3 (fine-tuning).
# OJO: tras el Experimento 4 la variable `model` fue liberada (del model);
# recuperamos el modelo fine-tuned desde el Trainer del Experimento 3.
mejor_modelo = trainer_exp3.model
mejor_modelo.eval()

resenas_demo = [
    "Excelente producto, llegó antes de lo esperado y funciona perfectamente. Muy recomendado.",
    "No sirve para nada, se dañó a la semana y el vendedor no responde. Pérdida de dinero.",
    "Está bien para el precio, aunque esperaba algo de mejor calidad. Cumple su función.",
    "La calidad es regular: tiene buenos detalles pero la batería dura muy poco.",
    "ME ENCANTÓ, superó todas mis expectativas, compraría de nuevo sin dudarlo!!",
]

for r in resenas_demo:
    pred, probs = predecir_estrellas(r, mejor_modelo, tokenizer)
    barra = ' '.join(f"{id2label[i]}:{p:.2f}" for i, p in enumerate(probs))
    print(f"→ {pred+1}★  ({barra})")
    print(f"  \"{r[:90]}\"")
    print()

# **Conclusiones**

_(Los valores exactos quedan registrados en la tabla de comparación tras ejecutar el notebook. Los puntos que siguen son la narrativa de los hallazgos.)_

1. **El pre-entrenamiento importa — mucho.** Con el MISMO dataset donde una LSTM llegaba a ~0.52 y un Transformer desde cero a ~0.49, BETO superó a ambos **incluso congelado** (experimentos 1, 2 y 4) y dio un salto adicional con fine-tuning. La hipótesis que dejamos abierta al final del Taller 2 quedó confirmada: _los transformers brillan con pre-entrenamiento_.
2. **La cabeza compleja aportó poco (H2 matizada).** Igual que en el notebook guía con noticias, cambiar una capa lineal por un MLP propio apenas movió los números: con el encoder congelado, el cuello de botella no es el clasificador sino la falta de adaptación al dominio.
3. **El fine-tuning fue el gran salto.** Al descongelar el encoder, BETO ajustó su representación al vocabulario de las reseñas y consiguió la mejor calidad. Costo: más tiempo de entrenamiento y riesgo de sobreajuste (por eso, pocas épocas y learning rate diminuto).
4. **Deep learning no siempre es necesario.** La regresión logística sobre embeddings congelados (experimento 4) fue sorprendentemente competitiva con una fracción del costo de entrenamiento — tal como lo anticipa la guía en su discusión.
5. **El patrón de errores es consistente con los talleres anteriores (H3 confirmada).** La matriz de confusión y el análisis de distancia muestran que la gran mayoría de errores son de **±1 estrella**: las reseñas 2-4★ combinan elogios y quejas de forma genuinamente ambigua. La métrica de "accuracy tolerante (±1★)" es, probablemente, la que importaría en producción.
6. **Decisiones de diseño que aprendimos:** usar el MISMO tokenizador del checkpoint (regla de oro), elegir `MAX_LEN` con datos del EDA (128 vs 512 de la guía: 4x menos cómputo con >99% de cobertura), y replicar el muestreo exacto de los talleres previos para que la comparación sea válida.

## **Líneas de trabajo futuro**

- **Tratar el problema como ordinal**, no como 5 clases independientes (p. ej. pérdida ordinal o clasificación por regresión) para penalizar de forma distinta errores de 1 estrella vs de 4.
- **Modelos más modernos:** comparar BETO con RoBERTa en español (`PlanTL-GOB-ES/roberta-base-bne`) o modelos multilingües tipo DistilBERT, que además reducirían el costo de inferencia.
- **Ajuste fino de hiperparámetros:** barrer learning rates (1e-5 a 5e-5), épocas y weight decay con `optuna`, y probar *layer-wise learning rate decay* (congelar menos capas gradualmente).
- **Análisis de errores con LLM:** usar un modelo generativo para etiquetar automáticamente las reseñas ambiguas y construir un dataset de casos difíciles.

## **Referencias**

- Dataset: [SetFit/amazon_reviews_multi_es](https://huggingface.co/datasets/SetFit/amazon_reviews_multi_es)
- Modelo: [BETO — dccuchile/bert-base-spanish-wwm-cased](https://huggingface.co/dccuchile/bert-base-spanish-wwm-cased)
- Devlin et al. (2018). [BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding](https://arxiv.org/abs/1810.04805)
- Canete et al. (2020). [Spanish Pre-Trained BERT Model and Evaluation Data](https://arxiv.org/abs/2308.02976)
- [Documentación de Hugging Face Transformers — Trainer](https://huggingface.co/docs/transformers/main_classes/trainer)
- Entregas previas del grupo: `Taller_1/proyecto_fundamentos_NLP_LSTM.ipynb`, `Taller_2/proyecto_transformers_clasificacion.ipynb`